# ShieldOS: Real-Time Fraud & Event Decisioning Gateway

### An evidence-backed fraud model with bounded risk orchestration and a measured decision path

**Interview walkthrough:** authentic OpenML transactions → chronological validation → calibrated risk → explicit policy thresholds → ATO/SIM-swap scenarios → specialist signal fan-out → audit trace → 1,000-request latency test.

<div style="display:flex;gap:12px;flex-wrap:wrap;margin:18px 0">
  <div style="background:#eef5f3;color:#17324d;padding:12px 18px;border-radius:14px"><b>OBSERVED DATA</b><br>OpenML 1597</div>
  <div style="background:#f5efe5;color:#17324d;padding:12px 18px;border-radius:14px"><b>DECISION ROUTES</b><br>Approve · MFA · Block</div>
  <div style="background:#f2edf0;color:#17324d;padding:12px 18px;border-radius:14px"><b>CONTROL PLANE</b><br>Deterministic online path</div>
</div>

The notebook demonstrates an engineering and product decision system. It does not claim a live bank deployment.

## 1 · Evidence boundary and operating topology

| Demonstrated with observed public data | Demonstrated as an operational scenario |
|---|---|
| Imbalanced fraud ranking, chronological evaluation, calibration, threshold trade-offs, value capture, score drift | SIM change, device trust, IP reputation, impossible travel, idempotency, timeouts, case routing |

The OpenML data have no IMSI, IMEI, device, IP, customer, geography, or carrier-event fields. Those signals are introduced only in the three labelled scenario fixtures.

<pre style="background:#f7f5f2;border:1px solid #dde5e6;border-radius:16px;padding:18px;color:#17324d">
HTTPS event
    │
    ▼
Schema + idempotency ──► deterministic hard stops
    │                            │ critical hit
    ▼                            ▼
Feature snapshot       bounded signal evaluators
    │                   identity · velocity · network · context
    └────────────────────────────┬──────────────────────────────
                                 ▼
                    calibrated model probability
                                 ▼
                 versioned policy + rule precedence
                   ╱              │              ╲
              APPROVE         STEP-UP MFA      HARD BLOCK
                   ╲              │              ╱
                    signed audit / decision event
                                 ▼
                off-path investigation agents + human
</pre>

Generative AI may summarize allowlisted evidence after the decision. It has no permission to change the risk score, policy threshold, settlement action, or authoritative audit event.

In [1]:
from pathlib import Path
import json, platform, sys, time
import numpy as np
import pandas as pd
from IPython.display import HTML, display
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, brier_score_loss, precision_recall_curve,
    roc_auc_score
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from xgboost import XGBClassifier
import plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ROOT = next(
    path for path in (Path.cwd(), Path.cwd().parent)
    if (path / "main.py").exists()
)
DATA_PATH = ROOT / "data" / "creditcard_openml_1597_extract.csv.gz"
MANIFEST_PATH = ROOT / "data" / "creditcard_openml_1597_manifest.json"
RANDOM_STATE = 42
SLA_MS = 45.0

COLORS = {
    "canvas": "#F7F5F2", "card": "#FFFFFF", "ink": "#17324D",
    "slate": "#60758A", "teal": "#4F7C82", "blue": "#7189B8",
    "amber": "#D5A24E", "rose": "#C66B78", "green": "#6F9C86",
    "grid": "rgba(23,50,77,0.09)"
}

def style_figure(fig, title, height=430):
    fig.update_layout(
        title={"text": title, "x": 0.02, "xanchor": "left"},
        height=height, paper_bgcolor=COLORS["canvas"],
        plot_bgcolor=COLORS["canvas"], font={"family": "Inter, Aptos, Arial", "color": COLORS["ink"]},
        margin={"l": 60, "r": 30, "t": 80, "b": 55},
        legend={"orientation": "h", "y": 1.08, "x": 0},
        hoverlabel={"bgcolor": COLORS["card"], "font_color": COLORS["ink"]},
    )
    fig.update_xaxes(gridcolor=COLORS["grid"], zeroline=False)
    fig.update_yaxes(gridcolor=COLORS["grid"], zeroline=False)
    return fig

def show(fig):
    fig.show(config={"displayModeBar": False, "responsive": True})

versions = pd.DataFrame({
    "component": ["Python", "pandas", "scikit-learn", "XGBoost", "Plotly"],
    "version": [platform.python_version(), pd.__version__,
                __import__("sklearn").__version__,
                __import__("xgboost").__version__, plotly.__version__]
})
display(versions.style.hide(axis="index").set_properties(**{"color": COLORS["ink"]}))

component,version
Python,3.12.14
pandas,2.2.3
scikit-learn,1.7.2
XGBoost,3.0.5
Plotly,6.3.1


In [2]:
manifest = json.loads(MANIFEST_PATH.read_text())
df = pd.read_csv(DATA_PATH)
df["Class"] = df["Class"].astype("int8")
required = {"Time", "Amount", "Class", "_source_row_number", "sampling_weight",
            *[f"V{i}" for i in range(1, 29)]}
checks = {
    "required columns": required.issubset(df.columns),
    "unique source rows": df["_source_row_number"].is_unique,
    "binary target": set(df["Class"].unique()) == {0, 1},
    "non-negative amount": bool((df["Amount"] >= 0).all()),
    "finite features": bool(np.isfinite(df[[f"V{i}" for i in range(1, 29)] + ["Time", "Amount"]]).all().all()),
    "ordered source time": bool(df["Time"].is_monotonic_increasing),
}
quality = pd.DataFrame(
    [{"control": k, "status": "PASS" if v else "FAIL"} for k, v in checks.items()]
)
assert all(checks.values())

provenance = pd.DataFrame({
    "field": ["Source", "Source rows", "Source fraud rows", "Committed extract",
              "Source SHA-256", "Extract SHA-256"],
    "value": [
        "OpenML 1597 · version 1", f"{manifest['source_rows']:,}",
        f"{manifest['source_fraud_rows']:,}", f"{len(df):,} rows",
        manifest["source_sha256"], manifest["extract_sha256"]
    ]
})
display(HTML("<h3 style='color:#17324d'>Verified provenance</h3>"))
display(provenance.style.hide(axis="index").set_properties(**{"color": COLORS["ink"]}))
display(quality.style.hide(axis="index").map(
    lambda value: "color:#477a65;font-weight:700" if value == "PASS" else ""
))

field,value
Source,OpenML 1597 · version 1
Source rows,"284,807"
Source fraud rows,492
Committed extract,"3,000 rows"
Source SHA-256,fdaf12730dc1fc426f318b71349f24f5c5fd00aa1152940be7e7509ae3d89d2a
Extract SHA-256,cf7398ad15c1a99e5cb48f3c1affd85def53b128a34ce2ee1af5805904f2ed0f


control,status
required columns,PASS
unique source rows,PASS
binary target,PASS
non-negative amount,PASS
finite features,PASS
ordered source time,PASS


In [3]:
kpis = [
    ("Source transactions", manifest["source_rows"], ",.0f", COLORS["blue"]),
    ("Observed fraud events", manifest["source_fraud_rows"], ",.0f", COLORS["rose"]),
    ("Observed prevalence", manifest["source_fraud_prevalence"] * 100, ".4f", COLORS["amber"]),
    ("Extract rows", len(df), ",.0f", COLORS["teal"]),
]
fig = make_subplots(rows=1, cols=4, specs=[[{"type": "indicator"}] * 4])
for idx, (title, value, fmt, color) in enumerate(kpis, start=1):
    suffix = "%" if "prevalence" in title.lower() else ""
    fig.add_trace(go.Indicator(
        mode="number", value=value, title={"text": title},
        number={"valueformat": fmt, "suffix": suffix, "font": {"color": color, "size": 31}}
    ), row=1, col=idx)
style_figure(fig, "Authentic transaction evidence · not a synthetic label set", height=250)
show(fig)

In [4]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Class imbalance · logarithmic count", "Transaction amount · log scale")
)
counts = df.groupby("Class", observed=True).size().rename(index={0: "Legitimate", 1: "Fraud"})
fig.add_trace(go.Bar(
    x=counts.index, y=counts.values,
    marker_color=[COLORS["teal"], COLORS["rose"]],
    text=[f"{value:,}" for value in counts.values], textposition="outside"
), row=1, col=1)
for label, cls, color in [("Legitimate", 0, COLORS["teal"]), ("Fraud", 1, COLORS["rose"])]:
    fig.add_trace(go.Histogram(
        x=np.log1p(df.loc[df["Class"] == cls, "Amount"]), name=label,
        opacity=0.62, nbinsx=55, marker_color=color, histnorm="probability density"
    ), row=1, col=2)
fig.update_yaxes(type="log", row=1, col=1)
fig.update_xaxes(title_text="Observed class", row=1, col=1)
fig.update_xaxes(title_text="log(1 + amount)", row=1, col=2)
fig.update_yaxes(title_text="Rows", row=1, col=1)
fig.update_yaxes(title_text="Density", row=1, col=2)
style_figure(fig, "Why raw accuracy is the wrong fraud metric", height=440)
fig.update_layout(barmode="overlay")
show(fig)

## 2 · Chronological validation

The deployment question is: **can a model fitted on earlier events rank later events?** The primary split is therefore chronological by ordered row position:

- first 60% → model training;
- next 20% → probability calibration and policy selection;
- final 20% → untouched evaluation.

The extract includes all labelled frauds and a deterministic sample of legitimate events. Sampling weights restore source prevalence for population and policy metrics. This is a reproducible engineering demonstration, not a leaderboard benchmark.

In [5]:
df = df.sort_values(["Time", "_source_row_number"], kind="stable").reset_index(drop=True)
n = len(df)
train_end, cal_end = int(0.60 * n), int(0.80 * n)
train, cal, test = df.iloc[:train_end].copy(), df.iloc[train_end:cal_end].copy(), df.iloc[cal_end:].copy()

def split_row(name, frame):
    return {
        "split": name, "rows": len(frame), "frauds": int(frame["Class"].sum()),
        "weighted prevalence": np.average(frame["Class"], weights=frame["sampling_weight"]),
        "start hour": frame["Time"].min() / 3600, "end hour": frame["Time"].max() / 3600,
    }

split_table = pd.DataFrame([
    split_row("Train", train), split_row("Calibration", cal), split_row("Test", test)
])
assert (split_table["frauds"] > 0).all()
display(split_table.style.hide(axis="index").format({
    "weighted prevalence": "{:.4%}", "start hour": "{:.1f}", "end hour": "{:.1f}"
}))

hourly = df.assign(hour=(df["Time"] // 3600).astype(int)).groupby("hour").apply(
    lambda g: pd.Series({
        "weighted_volume": g["sampling_weight"].sum(),
        "weighted_fraud_rate": np.average(g["Class"], weights=g["sampling_weight"])
    }), include_groups=False
).reset_index()
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.12)
fig.add_trace(go.Scatter(
    x=hourly["hour"], y=hourly["weighted_volume"], fill="tozeroy",
    line={"color": COLORS["blue"], "width": 2}, name="Weighted volume"
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=hourly["hour"], y=hourly["weighted_fraud_rate"] * 100,
    line={"color": COLORS["rose"], "width": 2.5}, name="Weighted fraud rate"
), row=2, col=1)
for boundary, label in [(train["Time"].max() / 3600, "train → calibration"),
                        (cal["Time"].max() / 3600, "calibration → test")]:
    fig.add_vline(x=boundary, line_dash="dot", line_color=COLORS["slate"])
    fig.add_annotation(x=boundary, y=1.04, yref="paper", text=label, showarrow=False,
                       font={"size": 11, "color": COLORS["slate"]})
fig.update_yaxes(title_text="Weighted events", row=1, col=1)
fig.update_yaxes(title_text="Fraud rate %", row=2, col=1)
fig.update_xaxes(title_text="Elapsed hour", row=2, col=1)
style_figure(fig, "Temporal structure before modelling", height=540)
show(fig)

split,rows,frauds,weighted prevalence,start hour,end hour
Train,1800,358,0.2185%,0.0,32.2
Calibration,600,57,0.0925%,32.3,40.1
Test,600,77,0.1297%,40.1,47.9


In [6]:
FEATURES = [f"V{i}" for i in range(1, 29)] + ["log_amount", "hour_sin", "hour_cos"]

def design(frame):
    X = frame[[f"V{i}" for i in range(1, 29)]].copy()
    X["log_amount"] = np.log1p(frame["Amount"])
    phase = 2 * np.pi * ((frame["Time"] / 3600) % 24) / 24
    X["hour_sin"], X["hour_cos"] = np.sin(phase), np.cos(phase)
    return X

X_train, X_cal, X_test = design(train), design(cal), design(test)
y_train, y_cal, y_test = train["Class"], cal["Class"], test["Class"]
w_cal, w_test = cal["sampling_weight"], test["sampling_weight"]

fit_started = time.perf_counter()
logistic = Pipeline([
    ("scale", RobustScaler()),
    ("model", LogisticRegression(
        class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE
    )),
])
logistic.fit(X_train, y_train)
logistic_fit_s = time.perf_counter() - fit_started

scale_pos_weight = (y_train == 0).sum() / max(1, (y_train == 1).sum())
fit_started = time.perf_counter()
xgb = XGBClassifier(
    objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
    n_estimators=220, max_depth=4, learning_rate=0.05, min_child_weight=4,
    subsample=0.85, colsample_bytree=0.82, reg_lambda=3.0, reg_alpha=0.1,
    scale_pos_weight=scale_pos_weight, n_jobs=4, random_state=RANDOM_STATE
)
xgb.fit(X_train, y_train, verbose=False)
xgb_fit_s = time.perf_counter() - fit_started

log_cal = logistic.predict_proba(X_cal)[:, 1]
xgb_cal = xgb.predict_proba(X_cal)[:, 1]
raw_cal = 0.15 * log_cal + 0.85 * xgb_cal
isotonic = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1)
isotonic.fit(raw_cal, y_cal, sample_weight=w_cal)

log_test = logistic.predict_proba(X_test)[:, 1]
xgb_test = xgb.predict_proba(X_test)[:, 1]
raw_test = 0.15 * log_test + 0.85 * xgb_test
p_cal, p_test = isotonic.transform(raw_cal), isotonic.transform(raw_test)

def metric_row(name, rank_probs, probability_probs=None):
    probability_probs = rank_probs if probability_probs is None else probability_probs
    return {
        "model": name,
        "PR-AUC": average_precision_score(y_test, rank_probs, sample_weight=w_test),
        "ROC-AUC": roc_auc_score(y_test, rank_probs, sample_weight=w_test),
        "Brier": brier_score_loss(y_test, probability_probs, sample_weight=w_test),
    }

metrics = pd.DataFrame([
    metric_row("Balanced logistic", log_test),
    metric_row("XGBoost", xgb_test),
    metric_row("Blend + isotonic calibration", raw_test, p_test),
])
display(HTML(
    f"<div style='display:flex;gap:12px;margin:12px 0'>"
    f"<div style='background:#eef5f3;padding:12px 18px;border-radius:12px'><b>Logistic fit</b><br>{logistic_fit_s:.2f}s</div>"
    f"<div style='background:#eef1f7;padding:12px 18px;border-radius:12px'><b>XGBoost fit</b><br>{xgb_fit_s:.2f}s</div>"
    f"<div style='background:#f5efe5;padding:12px 18px;border-radius:12px'><b>Features</b><br>{len(FEATURES)}</div>"
    f"</div>"
))
display(metrics.style.hide(axis="index").format({"PR-AUC": "{:.4f}", "ROC-AUC": "{:.4f}", "Brier": "{:.6f}"}))

model,PR-AUC,ROC-AUC,Brier
Balanced logistic,0.3792,0.9777,0.043414
XGBoost,0.8108,0.9878,0.009818
Blend + isotonic calibration,0.8200,0.9861,0.004058


In [7]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Precision–recall on untouched time window", "Probability reliability")
)
for name, probs, color in [
    ("Logistic", log_test, COLORS["slate"]),
    ("XGBoost", xgb_test, COLORS["blue"]),
    ("Raw blend ranking", raw_test, COLORS["teal"]),
]:
    precision, recall, _ = precision_recall_curve(y_test, probs, sample_weight=w_test)
    fig.add_trace(go.Scatter(
        x=recall, y=precision, mode="lines", name=name,
        line={"color": color, "width": 2.4}
    ), row=1, col=1)

prob_true, prob_pred = calibration_curve(y_test, p_test, n_bins=8, strategy="quantile")
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode="lines", name="Perfect calibration",
    line={"color": COLORS["grid"], "dash": "dash"}
), row=1, col=2)
fig.add_trace(go.Scatter(
    x=prob_pred, y=prob_true, mode="lines+markers", name="Isotonic probability",
    line={"color": COLORS["teal"], "width": 2.6}, marker={"size": 8}
), row=1, col=2)
fig.update_xaxes(title_text="Recall", range=[0, 1], row=1, col=1)
fig.update_yaxes(title_text="Precision", range=[0, 1], row=1, col=1)
fig.update_xaxes(title_text="Mean predicted probability", row=1, col=2)
fig.update_yaxes(title_text="Observed fraud rate", row=1, col=2)
style_figure(fig, "Ranking quality and calibrated decision evidence", height=450)
show(fig)

## 3 · Policy is a separate product decision

Model probability is evidence, not the action. The calibration window sets a scenario policy under explicit constraints:

- no more than 0.5% of weighted transaction volume leaves straight-through approval;
- a hard-block candidate aims for at least 80% weighted precision;
- ambiguous high-risk events enter MFA instead of being silently declined;
- deterministic critical rules retain precedence over the model.

The costs and capacity below are illustrative interview assumptions. They are not measured Shuru or bank economics.

In [8]:
def weighted_quantile(values, q, weights):
    order = np.argsort(values)
    values, weights = np.asarray(values)[order], np.asarray(weights)[order]
    cumulative = np.cumsum(weights) / np.sum(weights)
    return float(values[np.searchsorted(cumulative, q, side="left")])

def binary_policy_stats(probs, labels, weights, threshold):
    pred = np.asarray(probs) >= threshold
    labels, weights = np.asarray(labels), np.asarray(weights)
    tp = weights[pred & (labels == 1)].sum()
    fp = weights[pred & (labels == 0)].sum()
    total_fraud = weights[labels == 1].sum()
    return {
        "threshold": float(threshold),
        "precision": float(tp / max(tp + fp, 1e-12)),
        "fraud_recall": float(tp / max(total_fraud, 1e-12)),
        "intervention_rate": float(weights[pred].sum() / weights.sum()),
    }

candidate_thresholds = np.unique(np.quantile(p_cal, np.linspace(0.75, 0.9995, 300)))
frontier = pd.DataFrame([
    binary_policy_stats(p_cal, y_cal, w_cal, threshold)
    for threshold in candidate_thresholds
])
mfa_probability = weighted_quantile(p_cal, 0.995, w_cal)
feasible_block = frontier[
    (frontier["precision"] >= 0.80) &
    (frontier["intervention_rate"] <= 0.0020) &
    (frontier["threshold"] > mfa_probability)
]
if feasible_block.empty:
    block_row = frontier.sort_values(
        ["precision", "fraud_recall"], ascending=[False, False]
    ).iloc[0]
    block_selection = "fallback: maximum calibration precision"
else:
    block_row = feasible_block.sort_values("fraud_recall", ascending=False).iloc[0]
    block_selection = "constraint satisfied"
block_probability = float(block_row["threshold"])
if block_probability <= mfa_probability:
    block_probability = min(0.999999, weighted_quantile(p_cal, 0.9995, w_cal))
if block_probability <= mfa_probability:
    mfa_probability = max(1e-9, block_probability * 0.75)

def risk_score(probability):
    return np.interp(
        np.asarray(probability),
        [0.0, mfa_probability, block_probability, 1.0],
        [0.0, 40.0, 75.0, 100.0]
    )

selected = binary_policy_stats(p_cal, y_cal, w_cal, block_probability)
selection = pd.DataFrame([
    {"policy point": "MFA entry", "probability": mfa_probability, "risk score": 40,
     "selection": "0.5% weighted intervention budget"},
    {"policy point": "Hard-block entry", "probability": block_probability, "risk score": 75,
     "selection": block_selection},
])
display(selection.style.hide(axis="index").format({"probability": "{:.6f}"}))

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=frontier["intervention_rate"] * 100,
    y=frontier["fraud_recall"] * 100,
    mode="lines+markers",
    marker={"size": 6, "color": frontier["precision"], "colorscale": [
        [0, COLORS["rose"]], [0.5, COLORS["amber"]], [1, COLORS["green"]]
    ], "colorbar": {"title": "Precision"}},
    line={"color": "rgba(79,124,130,0.35)", "width": 1},
    name="Candidate thresholds"
))
fig.add_trace(go.Scatter(
    x=[selected["intervention_rate"] * 100], y=[selected["fraud_recall"] * 100],
    mode="markers+text", text=["selected block point"], textposition="top center",
    marker={"size": 15, "color": COLORS["ink"], "symbol": "diamond"}, name="Selected"
))
fig.update_xaxes(title_text="Weighted volume intervened %")
fig.update_yaxes(title_text="Fraud events captured %")
style_figure(fig, "Policy frontier · capture versus customer and operations load", height=450)
show(fig)

policy point,probability,risk score,selection
MFA entry,0.004391,40,0.5% weighted intervention budget
Hard-block entry,0.235466,75,constraint satisfied


In [9]:
test_scores = risk_score(p_test)
routes = np.select(
    [test_scores < 40, test_scores <= 75],
    ["STATUS_APPROVED", "STATUS_TRIGGER_MFA"],
    default="STATUS_HARD_BLOCK"
)
routed = test[["Class", "Amount", "sampling_weight"]].copy()
routed["route"] = routes
routed["probability"] = p_test
routed["risk_score"] = test_scores

route_rows = []
for route, group in routed.groupby("route", observed=True):
    weight = group["sampling_weight"]
    weighted_events = weight.sum()
    fraud_events = (group["Class"] * weight).sum()
    route_rows.append({
        "route": route,
        "weighted events": weighted_events,
        "volume share": weighted_events / routed["sampling_weight"].sum(),
        "fraud events": fraud_events,
        "queue precision": fraud_events / max(weighted_events, 1e-12),
        "fraud amount": group.loc[group["Class"] == 1, "Amount"].sum(),
    })
route_table = pd.DataFrame(route_rows).sort_values("route")
display(route_table.style.hide(axis="index").format({
    "weighted events": "{:,.0f}", "volume share": "{:.3%}",
    "fraud events": "{:,.1f}", "queue precision": "{:.2%}", "fraud amount": "USD {:,.0f}"
}))

sources, targets, values, colors = [], [], [], []
class_nodes = ["Observed legitimate", "Observed fraud"]
route_nodes = ["APPROVE", "MFA", "HARD BLOCK"]
for cls, class_name in [(0, class_nodes[0]), (1, class_nodes[1])]:
    for route, route_name in [
        ("STATUS_APPROVED", "APPROVE"),
        ("STATUS_TRIGGER_MFA", "MFA"),
        ("STATUS_HARD_BLOCK", "HARD BLOCK"),
    ]:
        mask = (routed["Class"] == cls) & (routed["route"] == route)
        value = routed.loc[mask, "sampling_weight"].sum()
        if value > 0:
            sources.append(class_nodes.index(class_name))
            targets.append(2 + route_nodes.index(route_name))
            values.append(value)
            colors.append("rgba(198,107,120,0.42)" if cls else "rgba(79,124,130,0.28)")

fig = go.Figure(go.Sankey(
    arrangement="snap",
    node={"label": class_nodes + route_nodes, "pad": 24, "thickness": 18,
          "color": [COLORS["teal"], COLORS["rose"], COLORS["green"], COLORS["amber"], COLORS["rose"]]},
    link={"source": sources, "target": targets, "value": values, "color": colors}
))
style_figure(fig, "Untouched test flow · observed class to decision route", height=430)
show(fig)

route,weighted events,volume share,fraud events,queue precision,fraud amount
STATUS_APPROVED,"59,074",99.509%,12.0,0.02%,"USD 2,375"
STATUS_HARD_BLOCK,290,0.488%,63.0,21.74%,"USD 6,006"
STATUS_TRIGGER_MFA,2,0.003%,2.0,100.00%,USD 15


In [10]:
routed["window"] = pd.qcut(np.arange(len(routed)), 4, labels=["T1", "T2", "T3", "T4"])
window_rows = []
for window, group in routed.groupby("window", observed=True):
    weight = group["sampling_weight"]
    labels = group["Class"]
    row = {
        "window": str(window),
        "weighted prevalence": np.average(labels, weights=weight),
        "PR-AUC": average_precision_score(labels, group["probability"], sample_weight=weight),
    }
    for route, short in [
        ("STATUS_APPROVED", "approve"), ("STATUS_TRIGGER_MFA", "MFA"),
        ("STATUS_HARD_BLOCK", "block")
    ]:
        row[short] = weight[group["route"] == route].sum() / weight.sum()
    window_rows.append(row)
stability = pd.DataFrame(window_rows)
display(stability.style.hide(axis="index").format({
    "weighted prevalence": "{:.4%}", "PR-AUC": "{:.4f}",
    "approve": "{:.2%}", "MFA": "{:.2%}", "block": "{:.2%}"
}))

fig = make_subplots(rows=1, cols=2, subplot_titles=("Ranking stability", "Routing stability"))
fig.add_trace(go.Scatter(
    x=stability["window"], y=stability["PR-AUC"], mode="lines+markers",
    line={"color": COLORS["teal"], "width": 3}, marker={"size": 9}, name="PR-AUC"
), row=1, col=1)
for column, color in [("approve", COLORS["green"]), ("MFA", COLORS["amber"]), ("block", COLORS["rose"])]:
    fig.add_trace(go.Scatter(
        x=stability["window"], y=stability[column] * 100, mode="lines+markers",
        line={"color": color, "width": 2.4}, name=column
    ), row=1, col=2)
fig.update_yaxes(title_text="PR-AUC", row=1, col=1)
fig.update_yaxes(title_text="Weighted route share %", row=1, col=2)
fig.update_xaxes(title_text="Sequential test window", row=1, col=1)
fig.update_xaxes(title_text="Sequential test window", row=1, col=2)
style_figure(fig, "Temporal monitoring · performance and queue load can move separately", height=430)
show(fig)

window,weighted prevalence,PR-AUC,approve,MFA,block
T1,0.1514%,0.9551,99.86%,0.00%,0.14%
T2,0.2110%,0.1591,98.99%,0.01%,1.01%
T3,0.0979%,0.8015,99.92%,0.00%,0.08%
T4,0.0698%,0.0337,99.24%,0.01%,0.76%


## 4 · Bounded agentic control plane

| Component | Permission | Timeout / fallback |
|---|---|---|
| Identity and device evaluator | score trusted device and SIM-change evidence | 12 ms; force at least MFA |
| Velocity evaluator | score hourly saturation | 12 ms; force at least MFA |
| Network evaluator | score hydrated IP reputation | 12 ms; force at least MFA |
| Transaction-context evaluator | score amount, geo velocity, authentication age | 12 ms; force at least MFA |
| Policy router | apply critical-rule precedence and thresholds | unknown version routes to review |
| Investigation agents | create timeline, entity graph, and cited case brief | asynchronous; human decides |

The synchronous components exchange typed signals. They are deliberately bounded deterministic evaluators rather than free-form LLM agents. The multi-agent investigation path runs from the immutable decision event, minimizes PII, and cannot write to settlement.

In [11]:
sys.path.insert(0, str(ROOT))
from main import ShieldOSGateway, demo_payloads, run_benchmark

with ShieldOSGateway(pseudonym_key=b"notebook-demo-key-material-32bytes!!") as gateway:
    decisions = [gateway.decide(payload) for payload in demo_payloads()]

decision_table = pd.DataFrame([{
    "scenario": ["Clean checkout", "Model-led anomaly", "SIM-swap hard stop"][idx],
    "risk score": decision.aggregate_risk_score,
    "action": decision.action.value,
    "HTTP": decision.http_status,
    "vector": decision.fraud_vector,
    "hard stop": decision.hard_stop,
    "degraded": decision.degraded,
    "latency ms": decision.processing_time_ms,
    "reason codes": " · ".join(decision.reason_codes),
    "audit digest": decision.audit_trace_hash[:16] + "…",
} for idx, decision in enumerate(decisions)])
display(decision_table.style.hide(axis="index").format({"risk score": "{:.2f}", "latency ms": "{:.3f}"}))

trace_rows = []
for idx, decision in enumerate(decisions):
    scenario = decision_table.iloc[idx]["scenario"]
    offset = 0.0
    for span in decision.trace:
        trace_rows.append({
            "scenario": scenario, "stage": span.stage, "start": offset,
            "duration": max(span.duration_ms, 0.002), "outcome": span.outcome.value
        })
        offset += max(span.duration_ms, 0.002)
trace_df = pd.DataFrame(trace_rows)
fig = go.Figure()
outcome_colors = {"PASS": COLORS["teal"], "DEGRADED": COLORS["amber"], "BLOCK": COLORS["rose"]}
for _, row in trace_df.iterrows():
    fig.add_trace(go.Bar(
        x=[row["duration"]], y=[row["scenario"]], base=[row["start"]],
        orientation="h", marker_color=outcome_colors[row["outcome"]],
        name=row["stage"], text=row["stage"], textposition="inside",
        hovertemplate=f"{row['stage']}<br>{row['duration']:.4f} ms<extra></extra>",
        showlegend=False
    ))
fig.update_xaxes(title_text="Cumulative in-process latency · ms")
fig.update_yaxes(title_text="")
style_figure(fig, "Replayable execution traces for three explicit scenarios", height=390)
fig.update_layout(barmode="overlay")
show(fig)

scenario,risk score,action,HTTP,vector,hard stop,degraded,latency ms,reason codes,audit digest
Clean checkout,6.08,STATUS_APPROVED,200,CLEAN_OR_LOW_RISK,False,False,0.508,NO_MATERIAL_RISK_SIGNAL,03ad4c87e6fc619d…
Model-led anomaly,43.23,STATUS_TRIGGER_MFA,202,VELOCITY_BURST,False,False,0.248,VELOCITY_ELEVATED · DEVICE_TRUST_LOW · VELOCITY_BURST,15bc0e64a0a33cc9…
SIM-swap hard stop,100.00,STATUS_HARD_BLOCK,403,SIM_SWAP_ANOMALY,True,False,0.011,SIM_SWAP_ON_SENSITIVE_TRANSACTION,a29c59367d8f407b…


In [12]:
with ShieldOSGateway(pseudonym_key=b"notebook-benchmark-key-material!!") as gateway:
    benchmark = run_benchmark(gateway, request_count=1_000, concurrency=8)

benchmark_frame = pd.DataFrame([benchmark.model_dump()])
display(benchmark_frame[[
    "request_count", "concurrency", "p50_ms", "p95_ms", "p99_ms",
    "max_ms", "throughput_rps", "p99_sla_met"
]].style.hide(axis="index").format({
    "p50_ms": "{:.3f}", "p95_ms": "{:.3f}", "p99_ms": "{:.3f}",
    "max_ms": "{:.3f}", "throughput_rps": "{:,.0f}"
}))

latency_names = ["P50", "P95", "P99", "Maximum"]
latency_values = [benchmark.p50_ms, benchmark.p95_ms, benchmark.p99_ms, benchmark.max_ms]
fig = go.Figure(go.Bar(
    x=latency_names, y=latency_values,
    marker_color=[COLORS["green"], COLORS["teal"], COLORS["blue"], COLORS["amber"]],
    text=[f"{value:.2f} ms" for value in latency_values], textposition="outside"
))
fig.add_hline(
    y=SLA_MS, line_dash="dash", line_color=COLORS["rose"],
    annotation_text="45 ms decision budget", annotation_position="top right"
)
fig.update_yaxes(title_text="Milliseconds")
style_figure(fig, "1,000 fresh requests · warmed local compute path", height=410)
show(fig)
print(benchmark.benchmark_scope)

request_count,concurrency,p50_ms,p95_ms,p99_ms,max_ms,throughput_rps,p99_sla_met
1000,8,1.378,3.689,5.407,9.982,"5,081",True


In-process decision engine only; excludes TLS, network hops, external feature-store I/O, Kafka acknowledgement, and ledger latency.


## 5 · Production map and interview close

| Notebook / sandbox | Production responsibility |
|---|---|
| Pydantic request and response | FastAPI or gRPC edge contract |
| Scenario signals | Redis-backed Feast online feature snapshot |
| deterministic rules | versioned in-memory policy engine |
| fitted XGBoost | calibrated artifact exported to ONNX Runtime |
| thread fan-out | async I/O with bulkheads and deadlines |
| route string | transactional Kafka producer plus outbox |
| local response cache | distributed idempotency store |
| audit digest | signed append-only decision envelope |
| sequential-window check | feature, score, label, and queue monitors |

**Controls required before production:** mTLS and workload identity, feature freshness, replay by model/policy version, champion–challenger shadowing, circuit breakers, timeout budgets, transactional outbox, delayed-label monitoring, queue-capacity alarms, investigator override, and model/rule rollback.

### Five defensible interview statements

1. The model is evaluated on later transactions, not a random shuffled split.
2. Calibration and business routing are separate concerns.
3. Critical deterministic rules override model probability through explicit precedence.
4. Generative investigation agents may explain evidence; they cannot authorize settlement.
5. The latency number is reproducible and scoped to local compute, not presented as a distributed SLA certification.

### What this notebook does not claim

- no production proof of SIM-swap or ATO recall;
- no customer-level longitudinal validation;
- no causal claim from anonymized PCA features;
- no live carrier, bank, Kafka, Redis, or ledger integration;
- no claim that this local benchmark certifies a bank-scale end-to-end SLA.

The restraint is intentional: the demo shows how to own fraud loss, false positives, customer friction, investigator capacity, auditability, and latency as one decision product.